# 🎯 Batch Scoring — Retail Demand Forecasting

Scores all store-product combinations to produce demand predictions
and inventory risk rankings.

**Reads from:** `gold_ml_features`, saved model

**Writes to:** `gold_ml_predictions`, `gold_ml_summary`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count,
    sum as spark_sum, round as spark_round
)
from pyspark.ml.feature import VectorAssembler, StringIndexer
from synapse.ml.lightgbm import LightGBMRegressionModel

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
model = LightGBMRegressionModel.load('Files/models/demand_forecasting_lgbm')
features_df = spark.read.table('gold_ml_features')
print(f'Scoring {features_df.count():,} feature rows')

In [ ]:
# Prepare features (same pipeline as training)
numeric_features = [
    'transaction_count', 'avg_discount', 'avg_price', 'daily_revenue',
    'day_of_week', 'day_of_month', 'month', 'week_of_year', 'is_weekend',
    'demand_lag_1', 'demand_lag_7', 'revenue_lag_1',
    'unit_cost', 'margin_pct',
]

cat_cols = ['category', 'subcategory', 'region', 'store_format']
indexed_df = features_df
for c in cat_cols:
    idx_col = f'{c}_idx'
    indexer = StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid='keep')
    indexed_df = indexer.fit(indexed_df).transform(indexed_df)

all_features = numeric_features + [f'{c}_idx' for c in cat_cols]
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='skip')
model_df = assembler.transform(indexed_df)

In [ ]:
# Score all data
scored = model.transform(model_df)

predictions = (
    scored
    .withColumn('predicted_demand', spark_round(col('prediction'), 1))
    .withColumn('demand_vs_actual', spark_round(col('prediction') - col('daily_quantity'), 1))
    .withColumn('demand_signal',
        when(col('prediction') > col('daily_quantity') * 1.3, 'high_demand')
        .when(col('prediction') < col('daily_quantity') * 0.7, 'low_demand')
        .otherwise('stable')
    )
    .withColumn('scored_at', current_timestamp())
    .select(
        'store_id', 'product_id', 'txn_date',
        'category', 'subcategory', 'region', 'store_format',
        'daily_quantity', 'predicted_demand', 'demand_vs_actual',
        'daily_revenue', 'avg_price', 'avg_discount',
        'demand_signal',
        'scored_at'
    )
)

predictions.write.mode('overwrite').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count():,} rows')
print(f'\nDemand signal distribution:')
predictions.groupBy('demand_signal').count().orderBy('count', ascending=False).show()

In [ ]:
# Summary: per product-category demand outlook
summary = (
    predictions
    .groupBy('category', 'region')
    .agg(
        spark_round(avg('predicted_demand'), 1).alias('avg_predicted_demand'),
        spark_round(avg('daily_quantity'), 1).alias('avg_actual_demand'),
        spark_round(avg('demand_vs_actual'), 1).alias('avg_demand_gap'),
        spark_round(spark_sum('daily_revenue'), 0).alias('total_revenue'),
        count('*').alias('observations'),
        spark_sum(when(col('demand_signal') == 'high_demand', 1).otherwise(0)).alias('high_demand_days'),
        spark_sum(when(col('demand_signal') == 'low_demand', 1).otherwise(0)).alias('low_demand_days'),
    )
    .withColumn('demand_trend',
        when(col('avg_demand_gap') > 2, 'growing')
        .when(col('avg_demand_gap') < -2, 'declining')
        .otherwise('stable')
    )
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('total_revenue').desc())
)

summary.write.mode('overwrite').format('delta').saveAsTable('gold_ml_summary')
print(f'\nDemand summary: {summary.count()} category-region combos')
summary.show(10, truncate=False)

In [ ]:
spark.sql('OPTIMIZE gold_ml_predictions')
spark.sql('OPTIMIZE gold_ml_summary')
print('\nAll Gold ML tables optimized')

print('\n=== Scoring Summary ===')
print(f'Total predictions: {predictions.count():,}')
print(f'Categories scored: {summary.count()}')
print('\nGold tables created:')
print('  - gold_ml_features')
print('  - gold_ml_model_metrics')
print('  - gold_ml_feature_importance')
print('  - gold_ml_predictions')
print('  - gold_ml_summary')